In [ ]:
import os
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

import optuna
from optuna.samplers import TPESampler

import shap

from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    auc,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    brier_score_loss,
)
from sklearn.linear_model import LogisticRegression



warnings.filterwarnings("ignore")

data_fs_static_external = pd.read_csv("YOUR_PATH")
data_fs_static_train = pd.read_csv("YOUR_PATH")

# Patient identifier used for grouped nested cross-validation and clustered bootstrap CIs.
# IMPORTANT: do not include this identifier in feature_space.
GROUP_COL = "subject_reference"

scale = 'yes'
feature_space = ['feature_1', 'feature_2', ...]

X_train, y_train  = do_train_test_split(data_fs_static_train,feature_space,scale)


In [ ]:
N_TRIALS_PER_INNER_OPTUNA = 200
OUTER_SPLITS = 5
INNER_SPLITS = 5

N_NESTED_TRIALS = 20
RANDOM_SEEDS = [1000 + i for i in range(N_NESTED_TRIALS)]

N_BOOTSTRAPS = 2000
BOOT_ALPHA = 0.05

OUT_DIR = "outputs_nested_rf"
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print("Ready.")


# ============================
# Helpers
# ============================

def as_numpy(y) -> np.ndarray:
    return np.asarray(y).reshape(-1)

def safe_confusion(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[int, int, int, int]:
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return int(tn), int(fp), int(fn), int(tp)

def stable_auprc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    return float(auc(rec, prec))

def point_metrics(y_true: np.ndarray, y_prob: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    tn, fp, fn, tp = safe_confusion(y_true, y_pred)
    spec = tn / (tn + fp + 1e-12)
    sens = tp / (tp + fn + 1e-12)
    npv  = tn / (tn + fn + 1e-12)

    out = {
        "auc": float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else float("nan"),
        "auprc": stable_auprc(y_true, y_prob),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "specificity": float(spec),
        "sensitivity": float(sens),
        "npv": float(npv),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "brier": float(brier_score_loss(y_true, y_prob)),
        "tn": float(tn), "fp": float(fp), "fn": float(fn), "tp": float(tp),
    }
    return out

def thresholds_from_predictions(y_true: np.ndarray, y_prob: np.ndarray) -> Dict[str, float]:
    prec, rec, thresh = precision_recall_curve(y_true, y_prob)
    if len(thresh) == 0:
        return {"Default-0.5": 0.5, "MCC-optimal": 0.5, "Youden": 0.5}

    t_default = 0.5

    mcc_vals = [matthews_corrcoef(y_true, (y_prob >= t).astype(int)) for t in thresh]
    t_mcc = float(thresh[int(np.nanargmax(mcc_vals))])

    youden_vals = []
    for t in thresh:
        yp = (y_prob >= t).astype(int)
        tn, fp, fn, tp = safe_confusion(y_true, yp)
        sens = tp / (tp + fn + 1e-12)
        spec = tn / (tn + fp + 1e-12)
        youden_vals.append(sens + spec - 1)
    t_youden = float(thresh[int(np.nanargmax(youden_vals))])

    return {"Default-0.5": t_default, "MCC-optimal": t_mcc, "Youden": t_youden}

def _logit(p: np.ndarray) -> np.ndarray:
    eps = 1e-12
    p = np.clip(p, eps, 1 - eps)
    return np.log(p) - np.log(1 - p)

def calibration_in_the_large(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    z = _logit(y_prob).reshape(-1, 1)
    lr = LogisticRegression(penalty="none", solver="lbfgs", max_iter=2000)
    lr.fit(z, y_true)
    return float(lr.intercept_[0])

def calibration_slope(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    z = _logit(y_prob).reshape(-1, 1)
    lr = LogisticRegression(penalty="none", solver="lbfgs", max_iter=2000)
    lr.fit(z, y_true)
    return float(lr.coef_[0][0])


# ============================
# Cluster bootstrap (patient-level resampling)
# ============================

def _prepare_cluster_indices(groups: np.ndarray) -> Tuple[np.ndarray, List[np.ndarray]]:
    """Precompute the admission-row indices belonging to each patient."""
    groups = as_numpy(groups)
    if pd.isna(groups).any():
        raise ValueError("Patient identifiers contain missing values; grouped resampling requires complete IDs.")

    unique_groups = pd.unique(groups)
    rows_by_group = [np.flatnonzero(groups == g) for g in unique_groups]
    return np.asarray(unique_groups, dtype=object), rows_by_group

def cluster_bootstrap_indices(
    rng: np.random.Generator,
    rows_by_group: List[np.ndarray],
) -> np.ndarray:
    """Resample patients with replacement, retaining all admissions for each sampled patient."""
    n_groups = len(rows_by_group)
    sampled_group_positions = rng.integers(0, n_groups, size=n_groups, endpoint=False)
    return np.concatenate([rows_by_group[j] for j in sampled_group_positions])

def bootstrap_metric_distribution(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    y_pred: np.ndarray,
    groups: np.ndarray,
    metrics: List[str],
    n_boot: int,
    seed: int,
) -> pd.DataFrame:
    """Admission-level metrics with patient-clustered bootstrap confidence intervals."""
    y_true = as_numpy(y_true)
    y_prob = as_numpy(y_prob)
    y_pred = as_numpy(y_pred)
    groups = as_numpy(groups)

    if not (len(y_true) == len(y_prob) == len(y_pred) == len(groups)):
        raise ValueError("y_true, y_prob, y_pred, and groups must have identical lengths.")

    rng = np.random.default_rng(seed)
    _, rows_by_group = _prepare_cluster_indices(groups)

    out_rows = []
    for b in range(n_boot):
        idx = cluster_bootstrap_indices(rng, rows_by_group)
        m = point_metrics(y_true[idx], y_prob[idx], y_pred[idx])
        for metric in metrics:
            out_rows.append({"boot_id": b, "metric": metric, "boot_value": float(m[metric])})
    return pd.DataFrame(out_rows)


# ============================
# RandomForest + Optuna (requested search spaces)
# ============================

def build_rf(best_params: Dict, seed: int) -> RandomForestClassifier:
    return RandomForestClassifier(
        n_estimators=int(best_params["n_estimators"]),
        max_depth=int(best_params["max_depth"]),
        min_samples_split=int(best_params["min_samples_split"]),
        min_samples_leaf=int(best_params["min_samples_leaf"]),
        max_features=best_params["max_features"],    # "sqrt" | "log2" | None
        bootstrap=bool(best_params["bootstrap"]),
        class_weight="balanced",
        n_jobs=-1,
        random_state=seed,
    )

def make_objective_rf(
    X: pd.DataFrame,
    y: pd.Series,
    groups: np.ndarray,
    inner_splits: int,
    seed: int,
):
    y_np = as_numpy(y)
    groups_np = as_numpy(groups)

    if not (len(X) == len(y_np) == len(groups_np)):
        raise ValueError("X, y, and groups must have identical lengths in Optuna inner CV.")

    def objective(trial: optuna.trial.Trial) -> float:
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "max_depth": trial.suggest_int("max_depth", 5, 20),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 12),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
            "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
        }

        cv = StratifiedGroupKFold(n_splits=inner_splits, shuffle=True, random_state=seed)
        aucs = []
        for tr, va in cv.split(X, y_np, groups=groups_np):
            overlap = set(groups_np[tr]).intersection(set(groups_np[va]))
            if overlap:
                raise RuntimeError(
                    f"Patient leakage detected in Optuna inner CV: {len(overlap)} overlapping patient(s)."
                )
            if len(np.unique(y_np[va])) < 2:
                raise RuntimeError(
                    "An Optuna inner validation fold contains only one outcome class. "
                    "Reduce INNER_SPLITS or inspect the grouped class distribution."
                )

            model = build_rf(params, seed=seed)
            model.fit(X.iloc[tr], y_np[tr])
            p = model.predict_proba(X.iloc[va])[:, 1]
            aucs.append(roc_auc_score(y_np[va], p))
        return float(np.mean(aucs))

    return objective

def oof_predict_proba_fixed_params_rf(
    X: pd.DataFrame,
    y: np.ndarray,
    groups: np.ndarray,
    best_params: Dict,
    n_splits: int,
    seed: int,
) -> np.ndarray:
    y = as_numpy(y)
    groups = as_numpy(groups)
    if not (len(X) == len(y) == len(groups)):
        raise ValueError("X, y, and groups must have identical lengths for OOF prediction.")

    oof = np.full(len(y), np.nan, dtype=float)
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for tr, va in cv.split(X, y, groups=groups):
        overlap = set(groups[tr]).intersection(set(groups[va]))
        if overlap:
            raise RuntimeError(
                f"Patient leakage detected in threshold-selection OOF CV: {len(overlap)} overlapping patient(s)."
            )
        model = build_rf(best_params, seed=seed)
        model.fit(X.iloc[tr], y[tr])
        oof[va] = model.predict_proba(X.iloc[va])[:, 1]

    if np.isnan(oof).any():
        raise RuntimeError("OOF probabilities contain NaNs.")
    return oof


# ============================
# SHAP helpers (tree)
# ============================

def shap_for_rf(
    fitted_model: RandomForestClassifier,
    X_background: pd.DataFrame,
    X_explain: pd.DataFrame,
) -> Tuple[np.ndarray, float]:
    """
    Tree SHAP for RF. Background can be used for interventional in some SHAP versions;
    safest general approach: use TreeExplainer(model) and explain X_explain.
    Returns shap_values (n_explain, n_features) and expected_value scalar.
    """
    explainer = shap.TreeExplainer(fitted_model)
    sv = explainer.shap_values(X_explain)

    # For binary classification, shap returns list [class0, class1] in many versions
    if isinstance(sv, list):
        sv = sv[1] if len(sv) > 1 else sv[0]

    exp_val = explainer.expected_value
    if isinstance(exp_val, (list, np.ndarray)):
        # expected_value may be per-class; use positive class if available
        if len(np.array(exp_val).reshape(-1)) > 1:
            exp_val = float(np.array(exp_val).reshape(-1)[1])
        else:
            exp_val = float(np.array(exp_val).reshape(-1)[0])

    return np.asarray(sv, dtype=float), float(exp_val)


# ============================
# Nested CV runner
# ============================

@dataclass
class NestedCVResult:
    seed: int
    y_true: np.ndarray
    p: np.ndarray
    idx: np.ndarray
    groups: np.ndarray
    fold_id: np.ndarray
    y_pred_default: np.ndarray
    y_pred_mcc: np.ndarray
    y_pred_youden: np.ndarray
    thresholds_per_fold: List[Dict[str, float]]
    best_params_per_fold: List[Dict]
    shap_values: np.ndarray
    shap_expected_value: float
    feature_names: List[str]
    fold_diagnostics: pd.DataFrame

def run_nested_cv_once(
    X: pd.DataFrame,
    y: pd.Series,
    groups: pd.Series,
    outer_splits: int,
    inner_splits: int,
    n_trials_optuna: int,
    seed: int,
) -> NestedCVResult:
    y_np = as_numpy(y)
    groups_np = as_numpy(groups)

    if not (len(X) == len(y_np) == len(groups_np)):
        raise ValueError("X, y, and groups must have identical lengths.")
    if pd.isna(groups_np).any():
        raise ValueError(f"{GROUP_COL} contains missing values.")

    outer_cv = StratifiedGroupKFold(n_splits=outer_splits, shuffle=True, random_state=seed)

    y_true_all, p_all, idx_all, groups_all, fold_id_all = [], [], [], [], []
    pred_default_all, pred_mcc_all, pred_youden_all = [], [], []
    thresholds_per_fold: List[Dict[str, float]] = []
    best_params_per_fold: List[Dict] = []
    shap_all = []
    shap_exp_vals = []
    fold_diagnostics = []
    feature_names = list(X.columns)

    for fold, (tr_idx, te_idx) in enumerate(
        outer_cv.split(X, y_np, groups=groups_np), start=1
    ):
        X_tr, y_tr = X.iloc[tr_idx], y_np[tr_idx]
        X_te, y_te = X.iloc[te_idx], y_np[te_idx]
        groups_tr = groups_np[tr_idx]
        groups_te = groups_np[te_idx]

        train_patients = set(groups_tr.tolist())
        validation_patients = set(groups_te.tolist())
        overlap = train_patients.intersection(validation_patients)
        if overlap:
            raise RuntimeError(
                f"Patient leakage detected in outer fold {fold}: "
                f"{len(overlap)} patient(s) occur in both train and validation."
            )

        fold_diagnostics.append({
            "seed": seed,
            "fold": fold,
            "n_train_admissions": len(tr_idx),
            "n_validation_admissions": len(te_idx),
            "n_train_patients": len(train_patients),
            "n_validation_patients": len(validation_patients),
            "n_overlapping_patients": len(overlap),
        })

        # Inner Optuna tuning on outer-training data, grouped by patient.
        study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=seed))
        study.optimize(
            make_objective_rf(
                X=X_tr,
                y=pd.Series(y_tr, index=X_tr.index),
                groups=groups_tr,
                inner_splits=inner_splits,
                seed=seed,
            ),
            n_trials=n_trials_optuna,
            show_progress_bar=False,
        )
        best_params = study.best_params
        best_params_per_fold.append(best_params)

        # Fit tuned model on the complete outer-training partition.
        model = build_rf(best_params, seed=seed)
        model.fit(X_tr, y_tr)
        p_te = model.predict_proba(X_te)[:, 1]

        # SHAP on outer-validation partition.
        sv_te, exp_val = shap_for_rf(model, X_background=X_tr, X_explain=X_te)
        shap_all.append(sv_te)
        shap_exp_vals.append(exp_val)

        # Leakage-free threshold selection from grouped OOF predictions generated
        # exclusively within the outer-training partition.
        p_tr_oof = oof_predict_proba_fixed_params_rf(
            X=X_tr,
            y=y_tr,
            groups=groups_tr,
            best_params=best_params,
            n_splits=inner_splits,
            seed=seed,
        )
        thr = thresholds_from_predictions(y_tr, p_tr_oof)
        thresholds_per_fold.append(thr)

        y_pred_default = (p_te >= thr["Default-0.5"]).astype(int)
        y_pred_mcc = (p_te >= thr["MCC-optimal"]).astype(int)
        y_pred_youden = (p_te >= thr["Youden"]).astype(int)

        y_true_all.append(y_te)
        p_all.append(p_te)
        idx_all.append(X_te.index.to_numpy())
        groups_all.append(groups_te)
        fold_id_all.append(np.full(len(te_idx), fold, dtype=int))
        pred_default_all.append(y_pred_default)
        pred_mcc_all.append(y_pred_mcc)
        pred_youden_all.append(y_pred_youden)

    shap_pooled = np.vstack(shap_all)
    exp_val_pooled = float(np.mean(shap_exp_vals))

    return NestedCVResult(
        seed=seed,
        y_true=np.concatenate(y_true_all),
        p=np.concatenate(p_all),
        idx=np.concatenate(idx_all),
        groups=np.concatenate(groups_all),
        fold_id=np.concatenate(fold_id_all),
        y_pred_default=np.concatenate(pred_default_all),
        y_pred_mcc=np.concatenate(pred_mcc_all),
        y_pred_youden=np.concatenate(pred_youden_all),
        thresholds_per_fold=thresholds_per_fold,
        best_params_per_fold=best_params_per_fold,
        shap_values=shap_pooled,
        shap_expected_value=exp_val_pooled,
        feature_names=feature_names,
        fold_diagnostics=pd.DataFrame(fold_diagnostics),
    )


# ============================
# Patient/admission summaries + group alignment
# ============================

def summarize_admissions_per_patient(
    df: pd.DataFrame,
    cohort_name: str,
    group_col: str = GROUP_COL,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Summarize unique patients and the distribution of admissions per patient."""
    if group_col not in df.columns:
        raise KeyError(f"{group_col!r} is not present in cohort {cohort_name!r}.")
    if df[group_col].isna().any():
        raise ValueError(f"{cohort_name}: {group_col} contains missing values.")

    counts = df.groupby(group_col, dropna=False).size().rename("n_admissions")
    q = counts.quantile([0.25, 0.50, 0.75])
    summary = pd.DataFrame([{
        "cohort": cohort_name,
        "n_admissions": int(len(df)),
        "n_unique_patients": int(counts.size),
        "n_patients_with_recurrent_admissions": int((counts > 1).sum()),
        "pct_patients_with_recurrent_admissions": float(100 * (counts > 1).mean()),
        "recurrent_admissions_present": bool((counts > 1).any()),
        "admissions_per_patient_mean": float(counts.mean()),
        "admissions_per_patient_sd": float(counts.std(ddof=1)) if counts.size > 1 else 0.0,
        "admissions_per_patient_min": int(counts.min()),
        "admissions_per_patient_q1": float(q.loc[0.25]),
        "admissions_per_patient_median": float(q.loc[0.50]),
        "admissions_per_patient_q3": float(q.loc[0.75]),
        "admissions_per_patient_max": int(counts.max()),
    }])

    distribution = (
        counts.value_counts().sort_index().rename_axis("n_admissions")
        .rename("n_patients").reset_index()
    )
    distribution.insert(0, "cohort", cohort_name)
    return summary, distribution

def align_groups_to_model_rows(
    source_df: pd.DataFrame,
    X: pd.DataFrame,
    group_col: str = GROUP_COL,
) -> pd.Series:
    """Align patient identifiers from the source cohort to rows returned by preprocessing."""
    if group_col not in source_df.columns:
        raise KeyError(f"{group_col!r} not found in the source training dataframe.")
    if source_df[group_col].isna().any():
        raise ValueError(f"{group_col} contains missing values.")

    if X.index.equals(source_df.index):
        groups = source_df.loc[X.index, group_col].copy()
        groups.index = X.index
        return groups

    ambiguous_reset_index = (
        len(X) < len(source_df)
        and isinstance(X.index, pd.RangeIndex)
        and X.index.start == 0
        and X.index.step == 1
    )
    if (
        not ambiguous_reset_index
        and source_df.index.is_unique
        and X.index.isin(source_df.index).all()
    ):
        groups = source_df.loc[X.index, group_col].copy()
        groups.index = X.index
        return groups

    if len(source_df) == len(X):
        warnings.warn(
            "X_train does not retain a directly mappable source index. "
            "Assuming preprocessing preserved row order when aligning subject_reference. "
            "For maximum safety, preserve the original DataFrame index in do_train_test_split()."
        )
        return pd.Series(source_df[group_col].to_numpy(), index=X.index, name=group_col)

    raise ValueError(
        "Could not safely align subject_reference to X_train. "
        "Preserve the original row index through do_train_test_split(), "
        "or return patient identifiers alongside X/y."
    )

# Reviewer-requested cohort-level admission summaries.
cohort_summary_frames = []
cohort_distribution_frames = []
for _cohort_name, _df in [("internal", data_fs_static_train), ("external", data_fs_static_external)]:
    if GROUP_COL in _df.columns:
        _summary, _distribution = summarize_admissions_per_patient(_df, _cohort_name)
        cohort_summary_frames.append(_summary)
        cohort_distribution_frames.append(_distribution)
    else:
        print(f"Skipping patient/admission summary for {_cohort_name}: {GROUP_COL!r} not found.")

if cohort_summary_frames:
    cohort_patient_summary_df = pd.concat(cohort_summary_frames, ignore_index=True)
    cohort_patient_summary_path = os.path.join(OUT_DIR, "cohort_patient_admission_summary.csv")
    cohort_patient_summary_df.to_csv(cohort_patient_summary_path, index=False)
    print(f"Saved cohort patient/admission summary: {cohort_patient_summary_path}")
    print(cohort_patient_summary_df)

if cohort_distribution_frames:
    admissions_per_patient_distribution_df = pd.concat(cohort_distribution_frames, ignore_index=True)
    admissions_per_patient_distribution_path = os.path.join(OUT_DIR, "admissions_per_patient_distribution.csv")
    admissions_per_patient_distribution_df.to_csv(admissions_per_patient_distribution_path, index=False)
    print(f"Saved admissions-per-patient distribution: {admissions_per_patient_distribution_path}")


# ============================
# Run
# ============================

if not isinstance(y_train, pd.Series):
    y_train = pd.Series(y_train, index=X_train.index)

X = X_train.copy()
y = y_train.copy()
groups = align_groups_to_model_rows(data_fs_static_train, X, GROUP_COL)

if GROUP_COL in X.columns:
    raise ValueError(
        f"{GROUP_COL} is present in X. Remove patient identifiers from feature_space before modeling."
    )

if not (X.index.equals(y.index) and X.index.equals(groups.index)):
    y = y.reindex(X.index)
    groups = groups.reindex(X.index)

if y.isna().any() or groups.isna().any():
    raise ValueError("Could not align X, y, and subject_reference without missing values.")

print(
    f"Internal modeling cohort: {len(X)} admissions from "
    f"{groups.nunique()} unique patients; "
    f"{(groups.value_counts() > 1).sum()} patients have recurrent admissions."
)

all_trial_preds: List[NestedCVResult] = []

print(f"Running repeated patient-grouped nested CV (Random Forest) ({N_NESTED_TRIALS} trials)...")
for i, seed in enumerate(RANDOM_SEEDS, start=1):
    print(f"\n=== Trial {i}/{len(RANDOM_SEEDS)} | seed={seed} ===")
    res = run_nested_cv_once(
        X=X,
        y=y,
        groups=groups,
        outer_splits=OUTER_SPLITS,
        inner_splits=INNER_SPLITS,
        n_trials_optuna=N_TRIALS_PER_INNER_OPTUNA,
        seed=seed,
    )
    all_trial_preds.append(res)

    patient_fold_counts = (
        pd.DataFrame({"patient": res.groups, "fold": res.fold_id})
        .groupby("patient")["fold"].nunique()
    )
    if int(patient_fold_counts.max()) != 1:
        raise RuntimeError("A patient was assigned to more than one outer validation fold.")

print("\nNested CV runs complete.")

cv_group_diagnostics_df = pd.concat(
    [r.fold_diagnostics.assign(trial=i) for i, r in enumerate(all_trial_preds, start=1)],
    ignore_index=True,
)
cv_group_diagnostics_path = os.path.join(OUT_DIR, "grouped_cv_fold_diagnostics.csv")
cv_group_diagnostics_df.to_csv(cv_group_diagnostics_path, index=False)
print(f"Saved grouped-CV diagnostics: {cv_group_diagnostics_path}")
print(
    "Maximum number of overlapping patients in any outer fold:",
    int(cv_group_diagnostics_df["n_overlapping_patients"].max()),
)


# ============================
# Metrics + PATIENT-CLUSTER BOOTSTRAP CIs
# ============================

METRICS_TO_REPORT = [
    "auc", "auprc", "f1", "mcc",
    "accuracy", "precision", "recall", "specificity", "sensitivity", "npv",
    "brier"
]
RULES = ["Default-0.5", "MCC-optimal", "Youden"]

# Per-trial point estimates
trial_point_rows = []
for t, res in enumerate(all_trial_preds, start=1):
    rule_to_pred = {
        "Default-0.5": res.y_pred_default,
        "MCC-optimal": res.y_pred_mcc,
        "Youden": res.y_pred_youden,
    }
    for rule, y_pred in rule_to_pred.items():
        m = point_metrics(res.y_true, res.p, y_pred)
        trial_point_rows.append({"trial": t, "seed": res.seed, "rule": rule,
                                 **{k: m[k] for k in METRICS_TO_REPORT}})

trial_point_df = pd.DataFrame(trial_point_rows)
trial_point_path = os.path.join(OUT_DIR, "nested_rf_trial_point_metrics.csv")
trial_point_df.to_csv(trial_point_path, index=False)
print(f"Saved per-trial point estimates: {trial_point_path}")

# Patient-cluster bootstrap within each trial, then pool draws across trials per rule+metric
all_boot_rows = []
for t, res in enumerate(all_trial_preds, start=1):
    rule_to_pred = {
        "Default-0.5": res.y_pred_default,
        "MCC-optimal": res.y_pred_mcc,
        "Youden": res.y_pred_youden,
    }

    for rule, y_pred in rule_to_pred.items():
        stable_rule_seed_offset = {
            "Default-0.5": 101,
            "MCC-optimal": 202,
            "Youden": 303,
        }
        boot_seed = int(res.seed + stable_rule_seed_offset[rule])
        dist = bootstrap_metric_distribution(
            y_true=res.y_true,
            y_prob=res.p,
            y_pred=y_pred,
            groups=res.groups,
            metrics=METRICS_TO_REPORT,
            n_boot=N_BOOTSTRAPS,
            seed=boot_seed,
        )
        dist["trial"] = t
        dist["seed"] = res.seed
        dist["rule"] = rule
        all_boot_rows.append(dist)

boot_df = pd.concat(all_boot_rows, ignore_index=True)
boot_path = os.path.join(OUT_DIR, f"nested_rf_bootstrap_distributions_{N_BOOTSTRAPS}x{N_NESTED_TRIALS}.csv")
boot_df.to_csv(boot_path, index=False)
print(f"Saved bootstrap distributions (may be large): {boot_path}")

# Summarize pooled bootstrap distributions per rule+metric (distributional CI)
summary_rows = []
for rule in RULES:
    for metric in METRICS_TO_REPORT:
        g = boot_df[(boot_df["rule"] == rule) & (boot_df["metric"] == metric)]["boot_value"].to_numpy(dtype=float)

        ci_low = float(np.nanpercentile(g, 100 * (BOOT_ALPHA / 2)))
        ci_high = float(np.nanpercentile(g, 100 * (1 - BOOT_ALPHA / 2)))

        pe = float(trial_point_df[trial_point_df["rule"] == rule][metric].mean())

        summary_rows.append({
            "rule": rule,
            "metric": metric,
            "point_estimate_mean_over_trials": pe,
            "bootstrap_ci_low": ci_low,
            "bootstrap_ci_high": ci_high,
            "n_boot_total": int(len(g)),
        })

bootstrap_summary_df = pd.DataFrame(summary_rows)
bootstrap_summary_path = os.path.join(OUT_DIR, "nested_rf_bootstrap_CI_summary.csv")
bootstrap_summary_df.to_csv(bootstrap_summary_path, index=False)
print(f"Saved bootstrap CI summary: {bootstrap_summary_path}")
print(bootstrap_summary_df)


# ============================
# Calibration per trial + patient-cluster bootstrap pooled across trials
# ============================

cal_point_rows = []
cal_boot_rows = []

for t, res in enumerate(all_trial_preds, start=1):
    citl = calibration_in_the_large(res.y_true, res.p)
    slope = calibration_slope(res.y_true, res.p)
    cal_point_rows.append({"trial": t, "seed": res.seed, "citl": citl, "slope": slope})

    rng = np.random.default_rng(res.seed + 99_999)
    _, rows_by_group = _prepare_cluster_indices(res.groups)
    for b in range(N_BOOTSTRAPS):
        idx = cluster_bootstrap_indices(rng, rows_by_group)
        yb = res.y_true[idx]
        pb = res.p[idx]

        if len(np.unique(yb)) < 2:
            citl_b = float("nan")
            slope_b = float("nan")
        else:
            citl_b = float(calibration_in_the_large(yb, pb))
            slope_b = float(calibration_slope(yb, pb))

        cal_boot_rows.append({"trial": t, "seed": res.seed, "boot_id": b, "metric": "citl",
                              "boot_value": citl_b})
        cal_boot_rows.append({"trial": t, "seed": res.seed, "boot_id": b, "metric": "slope",
                              "boot_value": slope_b})

cal_point_df = pd.DataFrame(cal_point_rows)
cal_point_path = os.path.join(OUT_DIR, "nested_rf_calibration_trial_points.csv")
cal_point_df.to_csv(cal_point_path, index=False)

cal_boot_df = pd.DataFrame(cal_boot_rows)
cal_boot_path = os.path.join(OUT_DIR, f"nested_rf_calibration_bootstrap_{N_BOOTSTRAPS}x{N_NESTED_TRIALS}.csv")
cal_boot_df.to_csv(cal_boot_path, index=False)

cal_summary_rows = []
for metric in ["citl", "slope"]:
    g = cal_boot_df[cal_boot_df["metric"] == metric]["boot_value"].to_numpy(dtype=float)
    ci_low = float(np.nanpercentile(g, 100 * (BOOT_ALPHA / 2)))
    ci_high = float(np.nanpercentile(g, 100 * (1 - BOOT_ALPHA / 2)))
    pe = float(cal_point_df[metric].mean())
    cal_summary_rows.append({
        "metric": metric,
        "point_estimate_mean_over_trials": pe,
        "bootstrap_ci_low": ci_low,
        "bootstrap_ci_high": ci_high,
        "n_boot_total": int(len(g)),
    })

cal_summary_df = pd.DataFrame(cal_summary_rows)
cal_summary_path = os.path.join(OUT_DIR, "nested_rf_calibration_bootstrap_CI_summary.csv")
cal_summary_df.to_csv(cal_summary_path, index=False)

print(f"Saved calibration point estimates: {cal_point_path}")
print(f"Saved calibration bootstrap distributions: {cal_boot_path}")
print(f"Saved calibration bootstrap CI summary: {cal_summary_path}")
print(cal_summary_df)


# ============================
# SHAP aggregation across ALL trials (pooled outer-test samples from every trial)
# ============================

shap_stack = np.vstack([r.shap_values for r in all_trial_preds])  # (sum_n, n_features)
feature_names = all_trial_preds[0].feature_names

shap_mean = shap_stack.mean(axis=0)
shap_mean_abs = np.abs(shap_stack).mean(axis=0)

shap_summary_df = pd.DataFrame({
    "feature": feature_names,
    "mean_shap": shap_mean,
    "mean_abs_shap": shap_mean_abs,
}).sort_values("mean_abs_shap", ascending=False)

shap_path = os.path.join(OUT_DIR, "nested_rf_shap_aggregated_over_20_trials.csv")
shap_summary_df.to_csv(shap_path, index=False)

print(f"Saved aggregated SHAP: {shap_path}")
print(shap_summary_df.head(25))

print("\nDone.")
